In [1]:
import glob
import os
import numpy as np

from run import run_model, evaluation
from data_preprocessing.time_stretch import time_stretch_dataset
from data_preprocessing.single_concat import concatenate_tracks_single_genre
from data_preprocessing.cross_concat import concatenate_tracks_diff_genre
from data_preprocessing.baseline import baseline
from utils import plot_results, plot_beats

# Dataset Preparation
---
*This code downloads the processed datasets. The dataset will be downloaded under "./dataset" folder directory.*

**Baseline**: Original Pop Genre data from RWC Popular dataset. Data is segmented into 30 seconds.

**Linear Time Stretch**: Applies time-varying stretch to 30-second audio segments where playback speed decreases linearly from 1.0x to 0.5x.

**Single-Genre Concatenation**: Combines two songs in different BPM. Each songs are segmented into 15s, concatenated both audio and annotations.

**Cross-Genre Concatenation**: Combines songs from two different genres, Each songs are segmented into 15s, concatenated both audio and annotations. 

## Directory setup. 
Please change the data_home if you are running in your machine.

In [2]:
rwc_popular = "rwc_popular"
rwc_home = "/Users/yujinkim/Google Drive/My Drive/Course/FA25/MIR/Final/rwc_popular"

gtzan_genre = "gtzan_genre"
gtzan_home = "/Users/yujinkim/mir_datasets/gtzan_genre"

## Download

In [3]:
# Baseline. (Pop Genre, segmented into 30s)
baseline(rwc_popular, rwc_home)

# Linear Time Stretch. 
time_stretch_dataset(rwc_popular, rwc_home)

# Pop-Genre Concatenation
concatenate_tracks_single_genre(rwc_popular, rwc_home)

# Cross-Genre Concatenation
concatenate_tracks_diff_genre(rwc_popular, rwc_home, gtzan_genre, gtzan_home)

/Users/yujinkim/miniconda3/envs/mirfinal/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Generation complete.
Time stretched dataset created at dataset/linear_stretch/rwc_popular directory
Generating 50 synthetic files in ./dataset/single_genre/rwc_popular...
Generation complete.
Processing genre: blues...
Processing genre: classical...
Processing genre: country...
Processing genre: disco...
Processing genre: hiphop...
Processing genre: jazz...
Error processing pair 54 (jazz): Error opening <_io.BufferedReader name='/Users/yujinkim/mir_datasets/gtzan_genre/gtzan_genre/genres/jazz/jazz.00054.wav'>: Format not recognised.
Processing genre: metal...
Processing genre: pop...
Processing genre: reggae...
Error processing pair 86 (reggae): Missing beat annotations
Processing genre: rock...
Cross-dataset generation complete.


# Experiments
---

This runs the experiments, including loading audio and annotations, run three models (BeatNet, Beat This!, Madmom), save the plots for estimated beats and ground truth beats.


#### Baseline experiments. The results are being saved as 'baseline_results.json'

In [4]:
audio_dir = "dataset/baseline/audio"
annotations_dir = "dataset/baseline/annotations"

audio_files = sorted(glob.glob(f"{audio_dir}/*.wav"))
annotation_files = sorted(glob.glob(f"{annotations_dir}/*.npy"))

for audio_file, annotation_file in zip(audio_files, annotation_files):
    annotation = np.load(annotation_file) 
    
    # get estimated beats from each models
    beats_bt, beats_bn, beats_mm = run_model(audio_file)
    
    # Plot the estimated beats, and save it as a .png
    plot_beats(audio_file, annotation, beats_bt, beats_bn, beats_mm, "baseline", save=True) 
    
    # Save the evaluation results (F-measure, Cemgil, Continuity) as json
    file_id = os.path.basename(audio_file).replace(".wav", "")
    evaluation(file_id, annotation, beats_bt, beats_bn, beats_mm, json_file="baseline_results.json", save=True) 
    
print("Results are saved as baseline_results.json at './results' folder")

Results are saved as baseline_results.json at './results' folder


#### Linear time stretch experiments. The results are being saved as 'linear_stretch_results.json'

In [5]:
audio_dir = "dataset/linear_stretch/rwc_popular/audio"
annotations_dir = "dataset/linear_stretch/rwc_popular/annotations"

audio_files = sorted(glob.glob(f"{audio_dir}/*.wav"))
annotation_files = sorted(glob.glob(f"{annotations_dir}/*.npy"))

for audio_file, annotation_file in zip(audio_files, annotation_files):
    annotation = np.load(annotation_file)
    
    # get estimated beats from each models
    beats_bt, beats_bn, beats_mm = run_model(audio_file)
    
    # Plot the estimated beats, and save it as a .png
    plot_beats(audio_file, annotation, beats_bt, beats_bn, beats_mm, "linear_stretch", save=True)
    
    # Save the evaluation results (F-measure, Cemgil, Continuity) as json
    file_id = os.path.basename(audio_file).replace(".wav", "")
    evaluation(file_id, annotation, beats_bt, beats_bn, beats_mm, json_file="linear_stretch_results.json")
    
print("Results are saved as linear_stretch_results.json at './results' folder")

Results are saved as linear_stretch_results.json at './results' folder


#### Single genre tempo changes experiments. The results are being saved as 'single_genre_results.json'

In [6]:
audio_dir = "dataset/single_genre/rwc_popular/audio"
annotations_dir = "dataset/single_genre/rwc_popular/annotations"

audio_files = sorted(glob.glob(f"{audio_dir}/*.wav"))
annotation_files = sorted(glob.glob(f"{annotations_dir}/*.npy"))

for audio_file, annotation_file in zip(audio_files, annotation_files):
    annotation = np.load(annotation_file)
    
    # get estimated beats from each models
    beats_bt, beats_bn, beats_mm = run_model(audio_file)
    
    # Plot the estimated beats, and save it as a .png
    plot_beats(audio_file, annotation, beats_bt, beats_bn, beats_mm, "single_genre", save=True)
    
    # Save the evaluation results (F-measure, Cemgil, Continuity) as json
    file_id = os.path.basename(audio_file).replace(".wav", "")
    evaluation(file_id, annotation, beats_bt, beats_bn, beats_mm, json_file="single_genre_results.json")
    
print("Results are saved as single_genre_results.json at './results' folder")

Results are saved as single_genre_results.json at './results' folder


#### Cross genre tempo changes experiments. The results are being saved as 'cross_genre_results.json'

In [7]:
genres = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

for genre in genres:
    audio_dir = f"dataset/cross_genre/{genre}/{genre}_audio"
    annotations_dir = f"dataset/cross_genre/{genre}/{genre}_annotations"

    audio_files = sorted(glob.glob(f"{audio_dir}/*.wav"))
    annotation_files = sorted(glob.glob(f"{annotations_dir}/*.npy"))

    for audio_file, annotation_file in zip(audio_files, annotation_files):
        annotation = np.load(annotation_file)
        
        # get estimated beats from each models
        beats_bt, beats_bn, beats_mm = run_model(audio_file)
        
        # Plot the estimated beats, and save it as a .png
        plot_beats(audio_file, annotation, beats_bt, beats_bn, beats_mm, f"cross_genre_{genre}", save=True)

        # Save the evaluation results (F-measure, Cemgil, Continuity) as json
        file_id = os.path.basename(audio_file).replace(".wav", "")
        evaluation(file_id, annotation, beats_bt, beats_bn, beats_mm, json_file= f"cross_genre_{genre}_results.json")
    
print("Results are saved as cross_genre_results.json at './results' folder")

Results are saved as cross_genre_results.json at './results' folder


#### Creating Box Plots
This code creates the box plots for each methods. Each of the box pots will be saved at "./results/box_plots"

In [8]:
METHODS = ['baseline', 'linear_stretch', 'single_genre']
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

for method in METHODS:
    json_file_name = f'{method}_results.json'

    plot_results(json_file_name)
    
for genre in GENRES:
    json_file_name = f"cross_genre_{genre}_results.json"
    
    plot_results(json_file_name)    

# For Results and Discussion, please check the results.ipynb notebook 